# 데이콘 x BDA 제 2회 학습자 수료 예측 AI 경진대회
## **1) 코드 흐름 요약**
전체 프로세스는 전처리 → 베이스 모델 구축 → 2단계 보정의 흐름을 따른다.

* Preprocessing (전처리):

결측치를 <<MISSING>>으로 채우고, 빈도가 낮은 범주는 기타로 통합하여 데이터의 복잡도를 줄인다.

시간 입력 데이터를 구간화(Bucket)하고, 자격증 보유 여부(ADsP, SQLD 등)를 추출하는 피처 엔지니어링을 수행한다.

**Logistic Regression(LR)**용 원-핫 인코딩(OHE) 데이터와 CatBoost용 범주형 데이터를 각각 분리하여 준비한다.

* Base Model (LR & Flip-rate 전략):

LR을 베이스 모델로 삼아 예측 확률을 구한다.

일반적인 임계값(Threshold) 방식이 아니라, 하위 30%(FLIP_FIXED)의 확률을 가진 샘플만 0으로 바꾸고 나머지는 모두 1로 예측하는 '비율 고정' 전략을 사용한다.

1. Rescue/Demote (CatBoost):

LR의 예측 확률을 CatBoost의 입력 피처로 다시 활용한다.

LR이 0으로 예측했지만 CatBoost가 보기에 1일 가능성이 높은 샘플은 **Rescue(구출)**하고, 반대로 1이지만 0일 가능성이 높은 샘플은 **Demote(강등)**하여 1차 교정을 진행한다.

2. Multi-CB Correction (최종 보정):

서로 다른 목표로 튜닝된 4개의 CatBoost 모델(A, B, C, D)을 활용한다.

여러 모델의 결과를 교집합(Rescue)과 합집합(Demote)으로 조합하여 오답 리스크를 관리하며 최종 예측값을 결정한다.

## **2) 정리**
*  새롭게 알게 된 / 배울 점

비율 고정(Flip-rate) 전략: 분류 문제에서 0.5라는 고정 수치 대신, 데이터 분포 특성에 맞춰 일정 비율을 강제로 할당하는 방식이 성능 안정화에 기여할 수 있음을 배울 수 있다.

경계선 가중치(Boundary Weights): 결정 경계 근처의 모호한 데이터에 가중치를 주어 모델이 해당 구간을 더 정밀하게 학습하도록 유도하는 기법이 유용하다.

다단계 보정 로직: 단일 모델의 한계를 극복하기 위해 '구출'과 '강등'이라는 개념을 도입하여 예측치를 정교하게 깎아나가는 과정이 매우 체계적이다.

* 어려운 내용

2단걔에서 여러 모델의 확률값과 분위수(Quantile)를 복합적으로 사용하여 '거부권(Veto)'이나 '추가 강등(Extra Demote)' 규칙을 적용하는 부분은 데이터에 대한 깊은 이해가 필요한 고난도 작업이다.

## 주요 코드

In [ ]:
# 1. 특정 비율(flip_rate)만큼 하위 확률 샘플을 0으로 뒤집는 핵심 로직
def all1_flip_pred(prob, flip_rate):
    n = len(prob)
    k = int(round(n * float(flip_rate)))
    pred = np.ones(n, dtype=int)
    if k > 0:
        order = np.argsort(prob, kind="mergesort")
        pred[order[:k]] = 0 # 하위 k개만 0으로 설정
    return pred

In [ ]:
# 2. 결정 경계 근처 샘플에 가중치를 부여하는 지수 감쇠 함수
def make_boundary_weights(lr_prob, flip_rate, tau=0.02, alpha=3.0):
    _, _, boundary, _ = compute_lr_boundary(lr_prob, flip_rate)
    dist = np.abs(lr_prob - boundary)
    w = 1.0 + float(alpha) * np.exp(-dist / max(1e-9, float(tau))) # 경계 근처에 높은 가중치
    return w, boundary

In [ ]:
# 3. 여러 모델의 결과를 조합하여 최종 구출 및 강등 결정 (Ensemble)
rescue_final = np.intersect1d(R2_A, R2_B).astype(int) # 모델 A, B가 모두 구출에 찬성할 때
demote_final_base = np.union1d(np.union1d(D2_A, D2_B), D2_C).astype(int) # A, B, C 중 하나라도 강등 의견일 때